# Model Training & Hyperparameter Tuning
**COEN 330 - Applied Machine Learning**

Trains and tunes **five** classifiers on the loan-approval task. One section per model, each showing baseline → tuning → per-fold cross-validation.

**Validation:** 5-fold StratifiedKFold on the **training set only**. The test set is loaded once at the top but is not used until final evaluation to prevent data leakage.

**Leakage-safe:** the preprocessor lives *inside* a `Pipeline`, so the scaler/encoder
refits on each fold's training portion during CV (never on the whole training set first).
Each saved model is a full pipeline (raw DataFrame in → prediction out).

**Target:** `loan_status` (1 = approved, 0 = rejected). Positive class = approved,
the **minority (~22%)**, so we tune class_weight between no weighting and balanced weighting where supported and **select on F1-Score**.

**Reported metrics:** accuracy, balanced accuracy, precision, recall, F1, PR-AUC.

**Models:** Logistic Regression (baseline), SVM (RBF-kernel), Gaussian Naive Bayes, Random Forest,
Gradient Boosting (HistGradientBoosting). A Dummy classifier which always predicts the majority class is used a floor check, not one of the five models.

## Setup

In [1]:
import sys, json
import importlib
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))   # make src/ importable

import pandas as pd
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from preprocessing import load_data, split_data
import train
importlib.reload(train)   # pick up edits to src/train.py without restarting kernel
from train import TrainingSession, PRIMARY, evaluate_baseline, run_feature_set, save_best_demo_models
from utils import DATA_RAW, DATASET_FILE, MODELS_DIR, RESULTS_DIR, SEED

DATA_PATH = DATA_RAW / DATASET_FILE

## Load and Split Traint/Test

`X_train` / `X_test` are raw DataFrames. The split is deterministic (fixed seed), so
the (04) evaluation notebook reproduces the identical split.

In [2]:
df = load_data(DATA_PATH, add_engineered=False)
X_train, X_test, y_train, y_test = split_data(df)

print('='*60)
print(f'TRAIN: {len(X_train):,} rows   |   approved (1) = {y_train.mean():.1%}')
print(f'TEST : {len(X_test):,} rows    |   SET ASIDE, used ONLY in 04_evaluation')
print('='*60)

TRAIN: 36,000 rows   |   approved (1) = 22.2%
TEST : 9,000 rows    |   SET ASIDE, used ONLY in 04_evaluation


## Training session

CV helpers and tuning logic live in `src/train.py`. The session accumulates
fitted pipelines and CV rows as each model cell runs below.

In [3]:
session = TrainingSession(X_train, y_train)

## Dummy Baseline (Floor Model)

Always predicts the majority class `Rejected`.

In [4]:
_ = evaluate_baseline(
    DummyClassifier(strategy="most_frequent"),
    X_train, y_train,
    "Dummy Baseline: 5-fold training CV:",
)

  Dummy Baseline: 5-fold training CV:
    f1            mean=0.0000  std=0.0000
    pr_auc        mean=0.2222  std=0.0000
    precision     mean=0.0000  std=0.0000
    recall        mean=0.0000  std=0.0000
    accuracy      mean=0.7778  std=0.0000
    balanced_acc  mean=0.5000  std=0.0000


## 1. Logistic Regression  (Baseline Model)

Logistic Regression via `saga` for the minority class.

In [5]:
session.run_model(
    "LogisticRegression",
    LogisticRegression(solver="saga", max_iter=2000, random_state=SEED),
    {
        "C":            [0.001, 0.01, 0.1, 1, 10],
        "l1_ratio":     [0, 0.5, 1],
        "class_weight": [None, "balanced"],
    },
)

LogisticRegression
  Baseline  : 5-fold CV (means/std):
    f1            mean=0.7661  std=0.0064
    pr_auc        mean=0.8573  std=0.0036
    precision     mean=0.7751  std=0.0061
    recall        mean=0.7575  std=0.0099
    accuracy      mean=0.8972  std=0.0026
    balanced_acc  mean=0.8473  std=0.0049
  Best params: {'C': 1, 'class_weight': None, 'l1_ratio': 0.5}
  Tuned     : 5-fold CV (per fold):
    f1            mean=0.7663  std=0.0066  folds=[0.761 0.760 0.769 0.778 0.763]
    pr_auc        mean=0.8572  std=0.0035  folds=[0.854 0.852 0.859 0.862 0.859]
    precision     mean=0.7752  std=0.0065  folds=[0.777 0.768 0.769 0.786 0.776]
    recall        mean=0.7576  std=0.0103  folds=[0.746 0.752 0.770 0.770 0.751]
    accuracy      mean=0.8973  std=0.0026  folds=[0.896 0.894 0.897 0.902 0.897]
    balanced_acc  mean=0.8474  std=0.0051  folds=[0.842 0.844 0.852 0.855 0.844]
  f1    : baseline=0.7661 -> tuned=0.7663  (delta +0.0001)
  pr_auc: baseline=0.8573 -> tuned=0.8572  (delt

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...ver='saga'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__C': [0.001, 0.01, ...], 'clf__class_weight': [None, 'balanced'], 'clf__l1_ratio': [0, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'balanced_acc': 'balanced_accuracy', 'f1': 'f1', 'pr_auc': 'average_precision', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versio

## 2. SVM (RBF-kernel)

Margin-based model. Slowest model on this many rows, so the grid is small.

In [6]:
session.run_model(
    "SVM_RBF",
    SVC(kernel="rbf", random_state=SEED),
    {
        "C":            [0.1, 1, 10],
        "gamma":        ["scale", "auto"],
        "class_weight": [None, "balanced"],
    },
)

SVM_RBF
  Baseline  : 5-fold CV (means/std):
    f1            mean=0.8008  std=0.0079
    pr_auc        mean=0.9003  std=0.0057
    precision     mean=0.8515  std=0.0106
    recall        mean=0.7560  std=0.0107
    accuracy      mean=0.9164  std=0.0033
    balanced_acc  mean=0.8591  std=0.0054
  Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto'}
  Tuned     : 5-fold CV (per fold):
    f1            mean=0.8040  std=0.0059  folds=[0.795 0.799 0.810 0.807 0.809]
    pr_auc        mean=0.9033  std=0.0055  folds=[0.901 0.895 0.909 0.902 0.910]
    precision     mean=0.8585  std=0.0095  folds=[0.854 0.853 0.847 0.872 0.867]
    recall        mean=0.7561  std=0.0108  folds=[0.744 0.752 0.776 0.751 0.758]
    accuracy      mean=0.9181  std=0.0023  folds=[0.915 0.916 0.919 0.920 0.920]
    balanced_acc  mean=0.8602  std=0.0048  folds=[0.854 0.857 0.868 0.860 0.862]
  f1    : baseline=0.8008 -> tuned=0.8040  (delta +0.0032)
  pr_auc: baseline=0.9003 -> tuned=0.9033  (delta +0.0030)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__C': [0.1, 1, ...], 'clf__class_weight': [None, 'balanced'], 'clf__gamma': ['scale', 'auto']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'balanced_acc': 'balanced_accuracy', 'f1': 'f1', 'pr_auc': 'average_precision', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versioncha

## 3. Gaussian Naive Bayes (Probabilistic)

The independence assumption is broken by the one-hot encoded columns, so GaussianNB is expected to perform poorly for our feature set.

In [7]:
session.run_model(
    "GaussianNB",
    GaussianNB(),
    {
        'var_smoothing': [1e-9, 1e-8, 1e-7]
    }
)

GaussianNB
  Baseline  : 5-fold CV (means/std):
    f1            mean=0.6232  std=0.0027
    pr_auc        mean=0.8210  std=0.0040
    precision     mean=0.4526  std=0.0028
    recall        mean=1.0000  std=0.0000
    accuracy      mean=0.7312  std=0.0031
    balanced_acc  mean=0.8272  std=0.0020
  Best params: {'var_smoothing': 1e-07}
  Tuned     : 5-fold CV (per fold):
    f1            mean=0.6334  std=0.0030  folds=[0.636 0.636 0.634 0.628 0.634]
    pr_auc        mean=0.8209  std=0.0040  folds=[0.820 0.815 0.821 0.820 0.828]
    precision     mean=0.4645  std=0.0026  folds=[0.466 0.467 0.466 0.459 0.465]
    recall        mean=0.9951  std=0.0031  folds=[0.999 0.997 0.993 0.990 0.997]
    accuracy      mean=0.7440  std=0.0026  folds=[0.746 0.746 0.745 0.739 0.744]
    balanced_acc  mean=0.8337  std=0.0027  folds=[0.836 0.836 0.834 0.829 0.834]
  f1    : baseline=0.6232 -> tuned=0.6334  (delta +0.0102)
  pr_auc: baseline=0.8210 -> tuned=0.8209  (delta -0.0002)
  (0.8s)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...aussianNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__var_smoothing': [1e-09, 1e-08, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'balanced_acc': 'balanced_accuracy', 'f1': 'f1', 'pr_auc': 'average_precision', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-

## 4. Random Forest (Bagging Ensemble)

In [8]:
session.run_model(
    "RandomForest",
    RandomForestClassifier(random_state=SEED, n_jobs=1),
    {
        "n_estimators": [200, 400],
        "max_depth":    [None, 10, 20],
        "class_weight": [None, "balanced"],
    },
)

RandomForest
  Baseline  : 5-fold CV (means/std):
    f1            mean=0.8239  std=0.0084
    pr_auc        mean=0.9232  std=0.0045
    precision     mean=0.8880  std=0.0071
    recall        mean=0.7684  std=0.0102
    accuracy      mean=0.9270  std=0.0033
    balanced_acc  mean=0.8703  std=0.0057
  Best params: {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 400}
  Tuned     : 5-fold CV (per fold):
    f1            mean=0.8256  std=0.0110  folds=[0.821 0.814 0.839 0.816 0.838]
    pr_auc        mean=0.9258  std=0.0051  folds=[0.923 0.918 0.932 0.925 0.931]
    precision     mean=0.8035  std=0.0092  folds=[0.801 0.790 0.815 0.800 0.813]
    recall        mean=0.8489  std=0.0136  folds=[0.843 0.839 0.866 0.833 0.864]
    accuracy      mean=0.9203  std=0.0048  folds=[0.918 0.915 0.926 0.916 0.926]
    balanced_acc  mean=0.8948  std=0.0079  folds=[0.892 0.887 0.905 0.886 0.904]
  f1    : baseline=0.8239 -> tuned=0.8256  (delta +0.0017)
  pr_auc: baseline=0.9232 -> tune

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__class_weight': [None, 'balanced'], 'clf__max_depth': [None, 10, ...], 'clf__n_estimators': [200, 400]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'balanced_acc': 'balanced_accuracy', 'f1': 'f1', 'pr_auc': 'average_precision', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... 

## 5. Gradient Boosting (Boosting Ensemble)

In [9]:
session.run_model(
    "GradientBoosting",
    HistGradientBoostingClassifier(random_state=SEED),
    {
        'learning_rate': [0.05, 0.1], 
        'max_depth': [None, 6], 
        'max_iter': [200, 400],
        'class_weight': [None, 'balanced']
    },
)

GradientBoosting
  Baseline  : 5-fold CV (means/std):
    f1            mean=0.8327  std=0.0096
    pr_auc        mean=0.9347  std=0.0041
    precision     mean=0.8877  std=0.0086
    recall        mean=0.7842  std=0.0142
    accuracy      mean=0.9300  std=0.0037
    balanced_acc  mean=0.8779  std=0.0072
  Best params: {'class_weight': None, 'learning_rate': 0.1, 'max_depth': 6, 'max_iter': 200}
  Tuned     : 5-fold CV (per fold):
    f1            mean=0.8370  std=0.0106  folds=[0.828 0.823 0.848 0.835 0.850]
    pr_auc        mean=0.9365  std=0.0040  folds=[0.935 0.929 0.940 0.938 0.940]
    precision     mean=0.8933  std=0.0103  folds=[0.883 0.889 0.887 0.894 0.913]
    recall        mean=0.7875  std=0.0154  folds=[0.780 0.766 0.812 0.784 0.796]
    accuracy      mean=0.9319  std=0.0041  folds=[0.928 0.927 0.935 0.931 0.938]
    balanced_acc  mean=0.8803  std=0.0079  folds=[0.875 0.869 0.891 0.879 0.887]
  f1    : baseline=0.8327 -> tuned=0.8370  (delta +0.0043)
  pr_auc: baseline=0

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__class_weight': [None, 'balanced'], 'clf__learning_rate': [0.05, 0.1], 'clf__max_depth': [None, 6], 'clf__max_iter': [200, 400]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'balanced_acc': 'balanced_accuracy', 'f1': 'f1', 'pr_auc': 'average_precision', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies 

## 6. Comparison Table

Tuned models, sorted by Primary Metric F1-score

In [10]:
results = session.results_df()
results

,model,best_params,f1,pr_auc,accuracy,balanced_acc,precision,recall,f1_std,pr_auc_std,accuracy_std,balanced_acc_std,precision_std,recall_std
0,GradientBoosting,"{'class_weight': None, 'learning_rate': 0.1, '...",0.836989,0.936542,0.931861,0.880304,0.893311,0.787500,0.010581,0.003966,0.004106,0.007851,0.010269,0.015376
1,RandomForest,"{'class_weight': 'balanced', 'max_depth': None...",0.825577,0.925761,0.920306,0.894795,0.803549,0.848875,0.010987,0.005076,0.004848,0.007906,0.009238,0.013598
2,SVM_RBF,"{'C': 10, 'class_weight': None, 'gamma': 'auto'}",0.803994,0.903332,0.918083,0.860241,0.858534,0.756125,0.005918,0.005489,0.002320,0.004773,0.009501,0.010763
3,LogisticRegression,"{'C': 1, 'class_weight': None, 'l1_ratio': 0.5}",0.766271,0.857250,0.897306,0.847420,0.775202,0.757625,0.006593,0.003542,0.002636,0.005058,0.006465,0.010318
4,GaussianNB,{'var_smoothing': 1e-07},0.633390,0.820876,0.744000,0.833688,0.464531,0.995125,0.003017,0.003992,0.002634,0.002699,0.002647,0.003147


## 7. Save Tuned Pipelines and CV Results

In [11]:
session.save()

## 8. Feature Set Experiments

Re-run four models (Logistic Regression, Random Forest, Gradient Boosting, Gaussian NB; SVM is excluded as it is too slow) on alternate feature sets and compare 5-fold CV `F1`.

Helpers live in `src/train.py`.

- **8.1**: without `previous_loan_defaults_on_file`
- **8.2**: add the two engineered ratio features (`employment_experience_ratio`, `credit_history_ratio`), with and without `previous_loan_defaults_on_file`

### 8.1 Run on WITHOUT `previous_loan_defaults_on_file`

In [12]:
res_without = run_feature_set(
    DATA_PATH,
    True,
    False,
    "WITHOUT_prevdef",
    save=True,
)

display(
    res_without.pivot_table(
        index="model",
        columns="feature_set",
        values=PRIMARY,
    ).round(4)
)

Feature set "WITHOUT_prevdef": tuning 4 models (LogisticRegression, RandomForest, GradientBoosting, GaussianNB)
  -> LogisticRegression ...
     done  F1=0.6109
  -> RandomForest ...
     done  F1=0.7672
  -> GradientBoosting ...
     done  F1=0.7735
  -> GaussianNB ...
     done  F1=0.6099


feature_set,WITHOUT_prevdef
model,
GaussianNB,0.6099
GradientBoosting,0.7735
LogisticRegression,0.6109
RandomForest,0.7672


### 8.2 Run WITH engineered ratio features

Adds `employment_experience_ratio` and `credit_history_ratio` 

(see `preprocessing.add_features`).

Trains both WITH and WITHOUT `previous_loan_defaults_on_file`.

In [13]:
res_eng_with = run_feature_set(
    DATA_PATH,
    False,
    True,
    "WITH_prevdef_eng",
    save=True,
)

res_eng_without = run_feature_set(
    DATA_PATH,
    True,
    True,
    "WITHOUT_prevdef_eng",
    save=True,
)

feature_set_results = pd.concat(
    [res_without, res_eng_with, res_eng_without],
    ignore_index=True,
)

feature_set_results.to_csv(
    RESULTS_DIR / "feature_set_comparison.csv",
    index=False,
)

display(
    feature_set_results.pivot_table(
        index="model",
        columns="feature_set",
        values=PRIMARY,
    ).round(4)
)

Feature set "WITH_prevdef_eng": tuning 4 models (LogisticRegression, RandomForest, GradientBoosting, GaussianNB)
  -> LogisticRegression ...
     done  F1=0.7676
  -> RandomForest ...
     done  F1=0.8275
  -> GradientBoosting ...
     done  F1=0.8361
  -> GaussianNB ...
     done  F1=0.6335
Feature set "WITHOUT_prevdef_eng": tuning 4 models (LogisticRegression, RandomForest, GradientBoosting, GaussianNB)
  -> LogisticRegression ...
     done  F1=0.6109
  -> RandomForest ...
     done  F1=0.7669
  -> GradientBoosting ...
     done  F1=0.7729
  -> GaussianNB ...
     done  F1=0.6091


feature_set,WITHOUT_prevdef,WITHOUT_prevdef_eng,WITH_prevdef_eng
model,,,
GaussianNB,0.6099,0.6091,0.6335
GradientBoosting,0.7735,0.7729,0.8361
LogisticRegression,0.6109,0.6109,0.7676
RandomForest,0.7672,0.7669,0.8275


### 8.3 Save best demo models

Copies to stable demo filenames:
- best main-session pipeline (WITH prevdef, section 7)
- best section 8.1 pipeline (WITHOUT prevdef)
- best section 8.2 pipeline (with engineered features: best performer among WITH vs WITHOUT prevdef)

In [14]:
save_best_demo_models(session, res_without, res_eng_with, res_eng_without)

Best WITH prevdef   : GradientBoosting  (F1=0.8370)
  -> D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\models\best_demo_model_with_prevdef.pkl
Best WITHOUT prevdef: GradientBoosting  (F1=0.7735)
  -> D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\models\best_demo_model_without_prevdef.pkl
Best engineered     : GradientBoosting  (WITH_prevdef_eng, F1=0.8361)
  -> D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\models\best_demo_model_engineered.pkl
Metadata -> D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\models\best_demo_model_metadata.json


{'with_prevdef': {'model_name': 'GradientBoosting',
  'feature_set': 'WITH_prevdef',
  'cross_validation_f1': 0.8369885916548272,
  'source_file': 'GradientBoosting.pkl',
  'demo_file': 'best_demo_model_with_prevdef.pkl'},
 'without_prevdef': {'model_name': 'GradientBoosting',
  'feature_set': 'WITHOUT_prevdef',
  'cross_validation_f1': 0.7734650509440071,
  'source_file': 'GradientBoosting_WITHOUT_prevdef.pkl',
  'demo_file': 'best_demo_model_without_prevdef.pkl'},
 'engineered': {'model_name': 'GradientBoosting',
  'feature_set': 'WITH_prevdef_eng',
  'cross_validation_f1': 0.8361227602681789,
  'source_file': 'GradientBoosting_WITH_prevdef_eng.pkl',
  'demo_file': 'best_demo_model_engineered.pkl'}}